# MNIST Handwritten Digit Recognition

Group Mini Project - Deep Learning

This notebook builds and compares neural networks that recognize handwritten digits (0-9) from the MNIST dataset, loaded directly from `keras.datasets`.
We train simple MLPs and CNNs, run a series of experiments (depth, regularization, data augmentation, optimizers, learning rate, batch size), and compare them by accuracy to find the best model.

## 1. Setup and Imports
Import libraries, fix random seeds for reproducibility.

In [1]:
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

## 2. Load and Explore the Data
Load MNIST from Keras (60,000 train / 10,000 test images of 28x28 grayscale digits), then inspect shapes, class distribution, and sample images.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Train:", x_train_full.shape, y_train_full.shape)
print("Test: ", x_test.shape, y_test.shape)
print("Pixel range:", x_train_full.min(), "-", x_train_full.max())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

counts = np.bincount(y_train_full)
axes[0].bar(range(10), counts)
axes[0].set_xticks(range(10))
axes[0].set_title("Training class distribution")
axes[0].set_xlabel("Digit")
axes[0].set_ylabel("Images")

axes[1].axis("off")
grid = np.concatenate([np.concatenate(list(x_train_full[r * 10:(r + 1) * 10]), axis=1) for r in range(3)], axis=0)
axes[1].imshow(grid, cmap="gray")
axes[1].set_title("Sample training images")
plt.tight_layout()
plt.show()

## 3. Preprocessing and Splitting
- Scale pixels to [0, 1] and add a channel dimension (28x28x1).
- Hold out 10% of the training set (stratified) as a validation set; the test set stays untouched until final evaluation.

In [ ]:
x_train_full = (x_train_full.astype("float32") / 255.0)[..., None]
x_test = (x_test.astype("float32") / 255.0)[..., None]

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, stratify=y_train_full, random_state=SEED
)

print("Train:", x_train.shape, "| Val:", x_val.shape, "| Test:", x_test.shape)

## 4. Helper Functions
Optimizer factory, a common training/evaluation routine (with EarlyStopping and ReduceLROnPlateau), and a plotting helper.
Every run is logged in a results list so all experiments can be compared at the end.

In [ ]:
results = []     # main experiments E1-E5
tuning = []      # hyperparameter / optimizer sweeps
histories = {}
models = {}


def make_optimizer(name, lr):
    name = name.lower()
    if name == "sgd":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    if name == "rmsprop":
        return keras.optimizers.RMSprop(learning_rate=lr)
    raise ValueError(f"Unknown optimizer: {name}")


def run_experiment(name, model, optimizer="adam", lr=1e-3, epochs=15, batch_size=128,
                   store=None, keep=True, verbose=0):
    store = results if store is None else store
    model.compile(
        optimizer=make_optimizer(optimizer, lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    cbs = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ]
    t0 = time.time()
    hist = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs, batch_size=batch_size,
        callbacks=cbs, verbose=verbose,
    )
    elapsed = time.time() - t0

    _, val_acc = model.evaluate(x_val, y_val, verbose=0)
    _, test_acc = model.evaluate(x_test, y_test, verbose=0)
    row = {
        "experiment": name, "optimizer": optimizer, "lr": lr, "batch_size": batch_size,
        "epochs_run": len(hist.history["loss"]), "params": model.count_params(),
        "val_acc": round(val_acc, 4), "test_acc": round(test_acc, 4), "time_s": round(elapsed, 1),
    }
    store.append(row)
    histories[name] = hist.history
    if keep:
        models[name] = model
    print(row)
    return model


def plot_history(names):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for n in names:
        h = histories[n]
        axes[0].plot(h["accuracy"], label=f"{n} train")
        axes[0].plot(h["val_accuracy"], "--", label=f"{n} val")
        axes[1].plot(h["loss"], label=f"{n} train")
        axes[1].plot(h["val_loss"], "--", label=f"{n} val")
    axes[0].set_title("Accuracy")
    axes[1].set_title("Loss")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Model Definitions
Builder functions for the architectures used in the experiments:
- `mlp_simple`: one hidden layer (baseline)
- `mlp_deep`: three hidden layers with dropout
- `cnn_basic`: one conv block + dense head
- `cnn_deep`: two conv blocks with BatchNorm and dropout, optional data augmentation

In [ ]:
def mlp_simple():
    return keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="mlp_simple")


def mlp_deep(dropout=0.3):
    return keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Flatten(),
        layers.Dense(512, activation="relu"), layers.Dropout(dropout),
        layers.Dense(256, activation="relu"), layers.Dropout(dropout),
        layers.Dense(128, activation="relu"), layers.Dropout(dropout),
        layers.Dense(10, activation="softmax"),
    ], name="mlp_deep")


def cnn_basic():
    return keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="cnn_basic")


def cnn_deep(dropout=0.3, augment=False):
    stack = [layers.Input((28, 28, 1))]
    if augment:  # these layers are only active during training
        stack += [
            layers.RandomRotation(0.05),
            layers.RandomTranslation(0.1, 0.1),
            layers.RandomZoom(0.1),
        ]
    stack += [
        layers.Conv2D(32, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"), layers.Dropout(dropout),
        layers.Dense(10, activation="softmax"),
    ]
    return keras.Sequential(stack, name="cnn_deep_aug" if augment else "cnn_deep")

## 6. Experiments E1-E5
Each experiment changes one thing relative to the previous one so the effect can be attributed:
- E1 baseline MLP -> E2 deeper MLP + dropout (depth and regularization)
- E3 basic CNN (effect of convolution) -> E4 deeper CNN + BatchNorm + dropout
- E5 best CNN + data augmentation (generalization)

In [ ]:
run_experiment("E1_mlp_simple", mlp_simple())
run_experiment("E2_mlp_deep", mlp_deep())
plot_history(["E1_mlp_simple", "E2_mlp_deep"])

In [ ]:
run_experiment("E3_cnn_basic", cnn_basic())
run_experiment("E4_cnn_deep", cnn_deep())
plot_history(["E3_cnn_basic", "E4_cnn_deep"])

In [ ]:
run_experiment("E5_cnn_deep_aug", cnn_deep(augment=True), epochs=25)
plot_history(["E4_cnn_deep", "E5_cnn_deep_aug"])

## 7. Optimizer and Hyperparameter Comparison
Short sweeps (8 epochs each) to see how optimizer, learning rate, batch size, and dropout rate affect results.
- Optimizers / learning rate / batch size are swept on `cnn_basic`.
- Dropout rate is swept on `cnn_deep`.

In [ ]:
EPOCHS = 8

# optimizers
for opt, lr in [("sgd", 1e-2), ("adam", 1e-3), ("rmsprop", 1e-3)]:
    run_experiment(f"T_opt_{opt}", cnn_basic(), optimizer=opt, lr=lr, epochs=EPOCHS, store=tuning, keep=False)

# learning rate (Adam)
for lr in [1e-2, 1e-3, 1e-4]:
    run_experiment(f"T_lr_{lr}", cnn_basic(), lr=lr, epochs=EPOCHS, store=tuning, keep=False)

# batch size (Adam)
for bs in [32, 128, 512]:
    run_experiment(f"T_bs_{bs}", cnn_basic(), batch_size=bs, epochs=EPOCHS, store=tuning, keep=False)

# dropout rate (deeper CNN)
for dr in [0.0, 0.3, 0.5]:
    run_experiment(f"T_dropout_{dr}", cnn_deep(dropout=dr), epochs=EPOCHS, store=tuning, keep=False)

pd.DataFrame(tuning)

In [ ]:
tuning_df = pd.DataFrame(tuning)
groups = ["T_opt", "T_lr", "T_bs", "T_dropout"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, g in zip(axes, groups):
    sub = tuning_df[tuning_df["experiment"].str.startswith(g)]
    ax.bar(sub["experiment"].str.replace(g + "_", "", regex=False), sub["val_acc"])
    ax.set_title(g.replace("T_", "").capitalize())
    ax.set_ylim(tuning_df["val_acc"].min() - 0.005, 1.0)
axes[0].set_ylabel("Validation accuracy")
plt.tight_layout()
plt.show()

## 8. Results Summary and Best Model
Compare all main experiments (accuracy, parameter count, training time). The best model is picked by **validation** accuracy; the test accuracy is only reported.

In [ ]:
summary = pd.DataFrame(results).sort_values("val_acc", ascending=False).reset_index(drop=True)
summary

In [ ]:
best_name = summary.loc[0, "experiment"]
best_model = models[best_name]
print("Best model:", best_name)

y_prob = best_model.predict(x_test, verbose=0)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion matrix - {best_name}")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f"{len(wrong)} misclassified out of {len(y_test)}")

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, idx in zip(axes.ravel(), wrong[:12]):
    ax.imshow(x_test[idx].squeeze(), cmap="gray")
    ax.set_title(f"true {y_test[idx]} / pred {y_pred[idx]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Optional: 5-Fold Cross-Validation
Check that the result of the best architecture is stable across folds. Disabled by default because it trains the model 5 times; set `RUN_CV = True` to run.

In [ ]:
RUN_CV = False
CV_EPOCHS = 5

if RUN_CV:
    builder = cnn_deep  # swap for the architecture of the best model
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_acc = []
    for fold, (tr, va) in enumerate(skf.split(x_train_full, y_train_full), 1):
        m = builder()
        m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        m.fit(x_train_full[tr], y_train_full[tr], epochs=CV_EPOCHS, batch_size=128, verbose=0)
        _, acc = m.evaluate(x_train_full[va], y_train_full[va], verbose=0)
        fold_acc.append(acc)
        print(f"Fold {fold}: {acc:.4f}")
    print(f"Mean: {np.mean(fold_acc):.4f} +/- {np.std(fold_acc):.4f}")

## 10. Conclusion
_Fill in after running the experiments:_
- Which architecture performed best, and by how much over the baseline MLP?
- What effect did convolution, regularization (dropout, BatchNorm), and augmentation have?
- Which optimizer / learning rate / batch size worked best?
- Typical mistakes (see misclassified digits) and possible future improvements.